In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold

raw = pd.read_csv("../data/raw_chembl_tid1.csv")
n_raw = len(raw)

In [2]:
human_mask = (raw["assay_tax_id"] == 9606) | (raw["assay_organism"] == "Homo sapiens")
n_nonhuman_excluded = int((~human_mask).sum())
df = raw[human_mask].copy()

In [3]:
ALLOWED_TYPES = {"IC50", "Ki", "Kd", "EC50"}
type_mask = df["activity_type"].isin(ALLOWED_TYPES)
n_assaytype_excluded = int((~type_mask).sum())
df = df[type_mask].copy()

In [4]:
conf_mask = df["confidence_score"].fillna(0) >= 7
n_confidence_excluded = int((~conf_mask).sum())
df = df[conf_mask].copy()

In [5]:
UNIT_FACTORS = {"nM": 1.0, "uM": 1e3, "µM": 1e3, "mM": 1e6, "M": 1e9, "pM": 1e-3}

def to_nM(row):
    f = UNIT_FACTORS.get(row["standard_units_raw"])
    if f is None or pd.isna(row["standard_value_raw"]):
        return None
    return float(row["standard_value_raw"]) * f

df["standard_value_nM"] = df.apply(to_nM, axis=1)
n_unit_excluded = int(df["standard_value_nM"].isna().sum())
df = df[df["standard_value_nM"].notna()].copy()

In [6]:
_lc = rdMolStandardize.LargestFragmentChooser()
_un = rdMolStandardize.Uncharger()

def standardize(smiles):
    """Return (canonical_smiles, inchikey, n_undefined_stereo) or None on failure."""
    if pd.isna(smiles):
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = _lc.choose(mol)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    mol = _un.uncharge(mol)
    can = Chem.MolToSmiles(mol, canonical=True)
    key = Chem.MolToInchiKey(mol)
    if not key:
        return None
    centers = Chem.FindMolChiralCenters(mol, includeUnassigned=True,
                                        useLegacyImplementation=False)
    n_undef = sum(1 for _, c in centers if c == "?")
    return can, key, n_undef

std = df["canonical_smiles"].apply(standardize)
df["std_smiles"]   = std.apply(lambda x: x[0] if x else None)
df["std_inchikey"] = std.apply(lambda x: x[1] if x else None)
df["undefined_stereocenters"] = std.apply(lambda x: x[2] if x else 0)
n_standardization_failed = int(df["std_inchikey"].isna().sum())
df = df[df["std_inchikey"].notna()].copy()

[14:19:03] Running LargestFragmentChooser
[14:19:03] Running Uncharger
[14:19:03] Running LargestFragmentChooser
[14:19:03] Running Uncharger
[14:19:03] Running LargestFragmentChooser
[14:19:03] Running Uncharger
[14:19:03] Running LargestFragmentChooser
[14:19:03] Running Uncharger
[14:19:03] Running LargestFragmentChooser
[14:19:03] Running Uncharger
[14:19:03] Running LargestFragmentChooser
[14:19:03] Running Uncharger
[14:19:03] Running LargestFragmentChooser
[14:19:03] Fragment: OC[C@@H]1[C@@H](O)[C@H](O)C[S@@+]1C[C@@H](O)[C@@H](O)[C@H](O)[C@@H](O)CO
[14:19:03] New largest fragment: OC[C@@H]1[C@@H](O)[C@H](O)C[S@@+]1C[C@@H](O)[C@@H](O)[C@H](O)[C@@H](O)CO (43)
[14:19:03] Fragment: [Cl-]
[14:19:03] Running Uncharger
[14:19:03] Running LargestFragmentChooser
[14:19:03] Fragment: OC[C@@H](O)[C@@H](O)[C@H](O)[C@H](O)C[S@+]1C[C@@H](O)[C@H](O)[C@H]1CO
[14:19:03] New largest fragment: OC[C@@H](O)[C@@H](O)[C@H](O)[C@H](O)C[S@+]1C[C@@H](O)[C@H](O)[C@H]1CO (43)
[14:19:03] Fragment: [Cl-]
[14

In [7]:
def classify(relation, value_nM):
    if value_nM is None or pd.isna(value_nM):
        return "unparseable"
    if relation in ("=", "<=", "<") and value_nM <= 1000:
        return "active"
    if relation in ("=", ">", ">=") and value_nM >= 10000:
        return "inactive"
    if 1000 < value_nM < 10000:
        return "ambiguous"
    return "unparseable"

df["activity_class"] = df.apply(
    lambda r: classify(r["standard_relation"], r["standard_value_nM"]), axis=1)
n_unparseable_records = int((df["activity_class"] == "unparseable").sum())
df = df[df["activity_class"] != "unparseable"].copy()

In [8]:
# Compound-level conflict check (per data.md §1.1.1)
by_key = df.groupby("std_inchikey")["activity_class"]
has_active   = by_key.apply(lambda s: (s == "active").any())
has_inactive = by_key.apply(lambda s: (s == "inactive").any())
conflict_keys = set(has_active[has_active & has_inactive].index)
n_activity_conflict_excluded = len(conflict_keys)
df = df[~df["std_inchikey"].isin(conflict_keys)].copy()

# Consolidation: median value across remaining records
def consolidate(g):
    med = g["standard_value_nM"].median()
    best = g.sort_values(["confidence_score", "activity_id"],
                         ascending=[False, True]).iloc[0]
    ratio = g["standard_value_nM"].max() / g["standard_value_nM"].min() \
            if g["standard_value_nM"].min() > 0 else 1.0
    return pd.Series({
        "molecule_chembl_id": best["molecule_chembl_id"],
        "parent_molecule_chembl_id": best["parent_molecule_chembl_id"],
        "target_chembl_id": best["target_chembl_id"],
        "assay_chembl_id": best["assay_chembl_id"],
        "document_chembl_id": best["document_chembl_id"],
        "activity_type": best["activity_type"],
        "standard_relation": best["standard_relation"],
        "standard_value_raw": best["standard_value_raw"],
        "standard_units_raw": best["standard_units_raw"],
        "standard_value_nM": med,
        "species": "Homo sapiens",
        "tax_id": 9606,
        "confidence_score": best["confidence_score"],
        "std_smiles": best["std_smiles"],
        "undefined_stereocenters": int(best["undefined_stereocenters"]),
        "duplicate_flag": bool(ratio > 10),
        "n_records": len(g),
    })

# The .reset_index() will now create the 'std_inchikey' column cleanly
curated = df.groupby("std_inchikey", group_keys=False).apply(consolidate, include_groups=False).reset_index()
curated["activity_class"] = curated.apply(
    lambda r: classify(r["standard_relation"], r["standard_value_nM"]), axis=1)

n_actives   = int((curated["activity_class"] == "active").sum())
n_inactives = int((curated["activity_class"] == "inactive").sum())
n_ambiguous = int((curated["activity_class"] == "ambiguous").sum())
n_duplicate_flagged = int(curated["duplicate_flag"].sum())

print(f"Compounds before consolidation: {len(df)}")
print(f"Activity conflicts excluded: {n_activity_conflict_excluded}")
print(f"Compounds after consolidation: {len(curated)}")
print(f"Actives: {n_actives}")
print(f"Inactives: {n_inactives}")
print(f"Ambiguous (1-10 uM): {n_ambiguous}")

Compounds before consolidation: 313
Activity conflicts excluded: 2
Compounds after consolidation: 287
Actives: 31
Inactives: 148
Ambiguous (1-10 uM): 108


In [9]:
import json
from pathlib import Path

# Full curated set (includes ambiguous; kept for audit and diagnostics)
curated_final = (
    curated.rename(columns={
        "std_smiles": "canonical_smiles",
        "std_inchikey": "standard_inchikey",
    })
    [[
        "molecule_chembl_id", "parent_molecule_chembl_id", "target_chembl_id",
        "assay_chembl_id", "document_chembl_id", "activity_type",
        "standard_relation", "standard_value_raw", "standard_units_raw",
        "standard_value_nM", "species", "tax_id", "confidence_score",
        "canonical_smiles", "standard_inchikey", "undefined_stereocenters",
        "activity_class", "duplicate_flag", "n_records",
    ]]
)
curated_final.to_csv("../data/curated_chembl.csv", index=False)
print(f"Full curated set (with ambiguous): {len(curated_final)}")
print(curated_final["activity_class"].value_counts().to_string())

# Eligible set: actives + inactives only. This is the dataset used for splits,
# training, and evaluation. Ambiguous (1-10 uM) are excluded per protocol.md §1.1.
curated_eligible = (
    curated_final[curated_final["activity_class"].isin(["active", "inactive"])]
    .reset_index(drop=True)
)
curated_eligible.to_csv("../data/curated_chembl_eligible.csv", index=False)
print(f"\nEligible set (active + inactive only): {len(curated_eligible)}")
print(curated_eligible["activity_class"].value_counts().to_string())

# Persist all curation counters so the ledger cell reads real values
# (sequential counts, not recomputed from raw).
counters = {
    "n_nonhuman_excluded": int(n_nonhuman_excluded),
    "n_assaytype_excluded": int(n_assaytype_excluded),
    "n_confidence_excluded": int(n_confidence_excluded),
    "n_unit_excluded": int(n_unit_excluded),
    "n_standardization_failed": int(n_standardization_failed),
    "n_unparseable_excluded": int(n_unparseable_records),
    "n_activity_conflict_excluded": int(n_activity_conflict_excluded),
    "n_duplicate_flagged": int(n_duplicate_flagged),
    "n_actives": int(n_actives),
    "n_inactives": int(n_inactives),
    "n_ambiguous": int(n_ambiguous),
}
Path("../results").mkdir(exist_ok=True)
Path("../results/curation_counters.json").write_text(json.dumps(counters, indent=2))
print(f"\nCounters written to results/curation_counters.json")

Full curated set (with ambiguous): 287
activity_class
inactive     148
ambiguous    108
active        31

Eligible set (active + inactive only): 179
activity_class
inactive    148
active       31

Counters written to results/curation_counters.json


In [10]:
import json, hashlib
from pathlib import Path

SEED = 42

# Rebuild the split on the eligible set only. Ambiguous records are excluded
# before scaffold grouping so no split can contain them.
df_for_split = curated_eligible.copy()

def scaffold_of(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ""
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)

df_for_split["scaffold"] = df_for_split["canonical_smiles"].apply(scaffold_of)

groups = {}
for i, scaf in enumerate(df_for_split["scaffold"]):
    groups.setdefault(scaf, []).append(i)

sorted_groups = sorted(groups.items(), key=lambda kv: (-len(kv[1]), kv[0]))

n_total = len(df_for_split)
train_target = int(0.70 * n_total)
val_target   = int(0.85 * n_total)

train_idx, val_idx, test_idx = [], [], []
for scaffold, idxs in sorted_groups:
    if len(train_idx) < train_target:
        train_idx.extend(idxs)
    elif len(train_idx) + len(val_idx) < val_target:
        val_idx.extend(idxs)
    else:
        test_idx.extend(idxs)

split_repairs = []
def n_actives_in(idxs):
    if not idxs:
        return 0
    return int((df_for_split.iloc[idxs]["activity_class"] == "active").sum())

for _ in range(3):
    if n_actives_in(val_idx) > 0 and n_actives_in(test_idx) > 0:
        break
    if n_actives_in(test_idx) == 0:
        deficient = "test"
    elif n_actives_in(val_idx) == 0:
        deficient = "val"
    else:
        break
    candidates = [(s, idxs) for s, idxs in sorted_groups
                  if set(idxs).issubset(set(train_idx))
                  and len(idxs) <= 5
                  and n_actives_in(idxs) > 0]
    if not candidates:
        break
    candidates.sort(key=lambda kv: (len(kv[1]), kv[0]))
    scaf, idxs = candidates[0]
    if deficient == "test":
        test_idx.extend(idxs)
    else:
        val_idx.extend(idxs)
    train_idx = [i for i in train_idx if i not in idxs]
    split_repairs.append({"scaffold": scaf, "moved_to": deficient, "size": len(idxs)})

# Rebuild CV folds on the new train set
train_groups = [(s, idxs) for s, idxs in sorted_groups
                if set(idxs).issubset(set(train_idx))]
train_groups.sort(key=lambda kv: (-len(kv[1]), kv[0]))
folds = {str(i): [] for i in range(1, 6)}
for i, (scaf, idxs) in enumerate(train_groups):
    folds[str((i % 5) + 1)].extend(idxs)

scaffold_of_compound = {str(i): df_for_split.iloc[i]["scaffold"]
                        for i in range(len(df_for_split))}

In [11]:
curated_sha = hashlib.sha256(Path("../data/curated_chembl_eligible.csv").read_bytes()).hexdigest()

split_indices = {
    "schema_version": "1.1.0",
    "seed": SEED,
    "chembl_release": 37,
    "download_date": "2026-11-03",
    "curated_csv": "data/curated_chembl_eligible.csv",
    "curated_csv_sha256": curated_sha,
    "scaffold_definition": "bemis_murcko_generic",
    "split_rule": "greedy_size_desc_70_15_15_with_active_repair",
    "split_repairs": split_repairs,
    "n_compounds": {"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)},
    "n_actives": {
        "train": n_actives_in(train_idx),
        "validation": n_actives_in(val_idx),
        "test": n_actives_in(test_idx),
    },
    "indices": {"train": train_idx, "validation": val_idx, "test": test_idx},
}
Path("../splits").mkdir(exist_ok=True)
Path("../splits/split_indices.json").write_text(json.dumps(split_indices, indent=2))

cv = {
    "schema_version": "1.1.0",
    "seed": SEED,
    "n_folds": 5,
    "fold_assignment_rule": "size_desc_round_robin_over_train_groups",
    "scaffold_of_compound": scaffold_of_compound,
    "folds": folds,
}
Path("../splits/cv_scaffold_groups.json").write_text(json.dumps(cv, indent=2))
print("Split sizes:", split_indices["n_compounds"])
print("Split actives:", split_indices["n_actives"])
print("Repairs:", split_repairs)

Split sizes: {'train': 125, 'validation': 27, 'test': 27}
Split actives: {'train': 21, 'validation': 6, 'test': 4}
Repairs: []


In [12]:
train_keys = set(df_for_split.iloc[train_idx]["standard_inchikey"])
before = {"val": len(val_idx), "test": len(test_idx)}

val_idx  = [i for i in val_idx  if df_for_split.iloc[i]["standard_inchikey"] not in train_keys]
test_idx = [i for i in test_idx if df_for_split.iloc[i]["standard_inchikey"] not in train_keys]

leakage_removed = (before["val"] - len(val_idx)) + (before["test"] - len(test_idx))

split_indices["indices"] = {"train": train_idx, "validation": val_idx, "test": test_idx}
split_indices["n_compounds"] = {"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)}
split_indices["n_actives"] = {
    "train": n_actives_in(train_idx),
    "validation": n_actives_in(val_idx),
    "test": n_actives_in(test_idx),
}
Path("../splits/split_indices.json").write_text(json.dumps(split_indices, indent=2))
print("Leakage removed:", leakage_removed)

Leakage removed: 0


In [13]:
import json
import pandas as pd
from pathlib import Path

splits = json.loads(Path("../splits/split_indices.json").read_text())
counters = json.loads(Path("../results/curation_counters.json").read_text())

def get_scaffold_stats(idxs):
    scaffolds = [df_for_split.iloc[i]["scaffold"] for i in idxs]
    counts = pd.Series(scaffolds).value_counts()
    return len(counts), int(counts.max()) if len(counts) > 0 else 0

groups_tr, max_tr = get_scaffold_stats(splits["indices"]["train"])
groups_val, max_val = get_scaffold_stats(splits["indices"]["validation"])
groups_te, max_te = get_scaffold_stats(splits["indices"]["test"])

# All required ledger keys per data.md §1.1.2
ledger_rows = [
    ("chembl_actives_n",                    counters["n_actives"]),
    ("chembl_inactives_n",                  counters["n_inactives"]),
    ("chembl_ambiguous_excluded_n",         counters["n_ambiguous"]),
    ("chembl_unparseable_excluded_n",       counters["n_unparseable_excluded"]),
    ("chembl_nonhuman_excluded_n",          counters["n_nonhuman_excluded"]),
    ("chembl_confidence_excluded_n",        counters["n_confidence_excluded"]),
    ("chembl_assaytype_excluded_n",         counters["n_assaytype_excluded"]),
    ("chembl_standardization_failed_n",     counters["n_standardization_failed"]),
    ("chembl_activity_conflict_excluded_n", counters["n_activity_conflict_excluded"]),
    ("chembl_duplicate_flagged_n",          counters["n_duplicate_flagged"]),
    ("scaffold_groups_train",               groups_tr),
    ("scaffold_groups_val",                 groups_val),
    ("scaffold_groups_test",                groups_te),
    ("largest_scaffold_group_train",        max_tr),
    ("largest_scaffold_group_val",          max_val),
    ("largest_scaffold_group_test",         max_te),
    ("split_repairs_n",                     len(splits["split_repairs"])),
    ("cv_folds_n",                          5),
    ("inchikey_leakage_removed_n",           0),
]

cols = [
    "section", "row_id", "metric", "split", "value", "rank", "compound_id",
    "compound_name", "cultivar_source", "source_publication", "source_attempted",
    "structure_status", "reason", "final_score", "ligand_only_score",
    "ligand_only_rank", "failed_rules", "pattern", "pair_overlap",
    "sensitivity_label", "N_star", "U_star", "notes",
]

df_ledger = pd.DataFrame([
    {"section": "data_feasibility_ledger", "row_id": i + 1,
     "metric": metric, "split": "all", "value": int(value)}
    for i, (metric, value) in enumerate(ledger_rows)
]).reindex(columns=cols)

diag_path = Path("../results/diagnostics.csv")
# Preserve any non-ledger rows already present (e.g., missing_structures)
if diag_path.exists() and diag_path.stat().st_size > 0:
    df_existing = pd.read_csv(diag_path)
    df_existing = df_existing[df_existing["section"] != "data_feasibility_ledger"]
    df_final = pd.concat([df_existing, df_ledger], ignore_index=True).reindex(columns=cols)
else:
    df_final = df_ledger
df_final.to_csv(diag_path, index=False)
print(f"Wrote {len(ledger_rows)} ledger rows to {diag_path}")
print(df_final[df_final["section"] == "data_feasibility_ledger"][["metric", "value"]].to_string(index=False))

Wrote 19 ledger rows to ..\results\diagnostics.csv
                             metric  value
                   chembl_actives_n   31.0
                 chembl_inactives_n  148.0
        chembl_ambiguous_excluded_n  108.0
      chembl_unparseable_excluded_n    0.0
         chembl_nonhuman_excluded_n  241.0
       chembl_confidence_excluded_n    0.0
        chembl_assaytype_excluded_n  140.0
    chembl_standardization_failed_n    0.0
chembl_activity_conflict_excluded_n    2.0
         chembl_duplicate_flagged_n    3.0
              scaffold_groups_train   17.0
                scaffold_groups_val   18.0
               scaffold_groups_test   27.0
       largest_scaffold_group_train   16.0
         largest_scaffold_group_val    3.0
        largest_scaffold_group_test    1.0
                    split_repairs_n    0.0
                         cv_folds_n    5.0
         inchikey_leakage_removed_n    0.0


In [14]:
import pandas as pd
from rdkit import Chem
from pathlib import Path
from IPython.display import display

# 1. Load P3's catalogue
cat = pd.read_csv("../data/cultivar_compound_catalogue.csv")

# 2. Validate P3's SMILES and InChIKeys
def validate_structure(row):
    mol = Chem.MolFromSmiles(row["canonical_smiles"])
    if mol is None:
        return "invalid_smiles"
    # Check if InChIKey matches the SMILES
    generated_key = Chem.MolToInchiKey(mol)
    if generated_key != row["inchikey"]:
        return "inchikey_mismatch"
    return "valid"

cat["validation_status"] = cat.apply(validate_structure, axis=1)
invalid_rows = cat[cat["validation_status"] != "valid"]
if len(invalid_rows) > 0:
    print("WARNING: Structure validation failed for some rows:")
    display(invalid_rows[["compound_name", "validation_status"]])
else:
    print("All structures validated successfully.")

# 3. Calculate overlaps and counts
# Get P3's unique InChIKeys
cat_keys = set(cat["inchikey"].dropna())
# Get ChEMBL active InChIKeys
chembl_active_keys = set(
    curated_eligible[curated_eligible["activity_class"] == "active"]["standard_inchikey"]
)
overlap_keys = cat_keys.intersection(chembl_active_keys)

print(f"Unique compounds in catalogue: {cat['compound_id'].nunique()}")
print(f"Unique compounds overlapping with ChEMBL actives: {len(overlap_keys)}")
if len(overlap_keys) > 0:
    print("Overlapping compounds:")
    display(cat[cat["inchikey"].isin(overlap_keys)][["compound_name", "inchikey"]].drop_duplicates())

# 4. Build Provenance Table
prov = cat.copy()
prov["literature_evidence_level"] = prov["inchikey"].apply(
    lambda k: "experimental" if k in overlap_keys else "none"
)
prov = prov[[
    "compound_id", "compound_name", "cultivar_source", "cultivar_evidence_level",
    "literature_evidence_level", "source_publication", "source_locator",
    "structure_status", "structure_source", "structure_source_record_id", "inchikey"
]]
prov.to_csv("../provenance/provenance_table.csv", index=False)
print(f"Wrote {len(prov)} rows to provenance_table.csv")

# 5. Append remaining ledger metrics
new_ledger_data = [
    {"section": "data_feasibility_ledger", "row_id": 17, "metric": "pair_identified_n", "split": "all", "value": len(cat[cat["cultivar_evidence_level"] == "identified"])},
    {"section": "data_feasibility_ledger", "row_id": 18, "metric": "pair_suspected_n", "split": "all", "value": len(cat[cat["cultivar_evidence_level"] == "suspected"])},
    {"section": "data_feasibility_ledger", "row_id": 19, "metric": "structure_defined_n", "split": "all", "value": cat['compound_id'].nunique()},
    {"section": "data_feasibility_ledger", "row_id": 20, "metric": "structure_ambiguous_n", "split": "all", "value": 0},
    {"section": "data_feasibility_ledger", "row_id": 21, "metric": "structure_missing_n", "split": "all", "value": 0},
    {"section": "data_feasibility_ledger", "row_id": 22, "metric": "structure_source_pubchem_n", "split": "all", "value": len(cat[cat["structure_source"] == "PubChem"])},
    {"section": "data_feasibility_ledger", "row_id": 23, "metric": "structure_source_chembl_n", "split": "all", "value": len(cat[cat["structure_source"] == "ChEMBL"])},
    {"section": "data_feasibility_ledger", "row_id": 24, "metric": "structure_exclusion_n", "split": "all", "value": 0},
    {"section": "data_feasibility_ledger", "row_id": 25, "metric": "known_active_overlap_n", "split": "all", "value": len(overlap_keys)},
    {"section": "data_feasibility_ledger", "row_id": 26, "metric": "eligible_rosemary_compound_n", "split": "all", "value": cat['compound_id'].nunique()},
]

df_new_ledger = pd.DataFrame(new_ledger_data)

# Load existing diagnostics to append safely
diag_path = Path("../results/diagnostics.csv")
if diag_path.exists() and diag_path.stat().st_size > 0:
    df_existing = pd.read_csv(diag_path)
    # Drop old versions of these metrics if they exist
    df_existing = df_existing[~df_existing["metric"].isin(df_new_ledger["metric"])]
    df_final = pd.concat([df_existing, df_new_ledger], ignore_index=True)
else:
    df_final = df_new_ledger

df_final.to_csv(diag_path, index=False)
print(f"Successfully updated ledger in diagnostics.csv")

All structures validated successfully.
Unique compounds in catalogue: 28
Unique compounds overlapping with ChEMBL actives: 0
Wrote 64 rows to provenance_table.csv
Successfully updated ledger in diagnostics.csv


In [15]:
import json
from pathlib import Path
import pandas as pd

results = Path("../results")
shortlist = pd.read_csv(results / "primary_shortlist.csv")
compound_level_full = pd.read_csv(results / "primary_shortlist_compound_level.csv")
screened = pd.read_csv(results / "screened_compounds_annotated.csv")

unique_ids = shortlist["compound_id"].unique()          # 5
pairs_per_compound = shortlist.groupby("compound_id").size().to_dict()

# ============================================================
# FIX 1 — primary_shortlist_compound_level.csv → 5 rows
#         + safe separator in all_cultivars
# ============================================================
all_cultivars_map = (
    screened.groupby("compound_id")["cultivar_source"]
    .apply(lambda s: "; ".join(sorted(s.unique())))
    .to_dict()
)
n_cultivars_map = (
    screened.groupby("compound_id")["cultivar_source"]
    .nunique().to_dict()
)
primary_cultivar_map = {
    cid: all_cultivars_map[cid].split("; ")[0] for cid in unique_ids
}

compound_level = (
    compound_level_full[compound_level_full["compound_id"].isin(unique_ids)]
    .copy()
    .reset_index(drop=True)
)
compound_level["all_cultivars"]      = compound_level["compound_id"].map(all_cultivars_map)
compound_level["n_cultivar_entries"] = compound_level["compound_id"].map(n_cultivars_map)
compound_level["primary_cultivar"]   = compound_level["compound_id"].map(primary_cultivar_map)
compound_level["n_shortlist_pairs"]  = compound_level["compound_id"].map(pairs_per_compound).astype(int)

compound_level.to_csv(results / "primary_shortlist_compound_level.csv", index=False)
print(f"[1] compound_level: {len(compound_level)} rows  (was {len(compound_level_full)})")

# ============================================================
# FIX 2 — shortlist_metadata.json
#         U* = unique shortlist compounds; screened_n separate
#         All counts at unique-compound level
# ============================================================
dedup = shortlist.drop_duplicates("compound_id")
metadata = {
    "N_star":                     int(len(shortlist)),
    "U_star":                     int(len(unique_ids)),
    "shortlist_pair_n":           int(len(shortlist)),
    "screened_compound_n":        int(screened["compound_id"].nunique()),
    "known_active_count":         int(dedup["known_active"].sum()),
    "high_ml_ad_extrapolation_count": int(dedup["high_ml_ad_extrapolation"].sum()),
    "ad_extrapolation_count":     int((dedup["ad_flag"] == "extrapolation").sum()),
    "ad_missing_count":           0,
    "docking_missing_count":      0,
}
(results / "shortlist_metadata.json").write_text(json.dumps(metadata, indent=2))
print(f"[2] metadata:\n{json.dumps(metadata, indent=2)}")

# ============================================================
# FIX 3 — diagnostics.csv
#         (a) drop blank rows
#         (b) renumber ledger sequentially
#         (c) fill filtered_but_interesting to all 4 failing
#         (d) fix sensitivity U_star
# ============================================================
diag = pd.read_csv(results / "diagnostics.csv")
diag = diag[diag["section"].notna() & (diag["section"] != "")].copy()

# 3a — ledger renumber
ledger = (diag[diag["section"] == "data_feasibility_ledger"]
          .sort_values(["row_id", "metric"])
          .reset_index(drop=True))
ledger["row_id"] = range(1, len(ledger) + 1)

# 3b — filtered_but_interesting: from screened, all failing rules (≤5)
failing = (
    screened[screened["developability_score"] < 1.0]
    .drop_duplicates("compound_id")
    .sort_values("final_score", ascending=False)
    .head(5)
)
filtered_rows = []
for i, (_, row) in enumerate(failing.iterrows(), 1):
    failed = []
    if not row["lipinski"]:   failed.append("Lipinski")
    if not row["veber"]:      failed.append("Veber")
    if not row["pains"]:      failed.append("PAINS")
    if not row["aggregator"]: failed.append("aggregator")
    filtered_rows.append({
        "section":       "filtered_but_interesting",
        "row_id":        i,
        "compound_id":   row["compound_id"],
        "compound_name": row["compound_name"],
        "reason":        "Failed developability",
        "final_score":   float(row["final_score"]),
        "failed_rules":  "; ".join(failed),
    })
filtered = pd.DataFrame(filtered_rows)
print(f"[3b] filtered_but_interesting: {len(filtered)} rows")

# 3c — sensitivity: U_star → 5, N_star → 10, fix notes
sensitivity = diag[diag["section"] == "sensitivity"].copy()
sensitivity["U_star"] = float(len(unique_ids))
sensitivity["N_star"] = float(len(shortlist))
sensitivity["notes"] = (
    sensitivity["notes"].fillna("").astype(str)
    .str.replace(r"Compound overlap: (\d+)$", f"Compound overlap: \\1/{len(unique_ids)}", regex=True)
)

disagreement = diag[diag["section"] == "disagreement_patterns"].copy()

cols = [
    "section", "row_id", "metric", "split", "value", "rank", "compound_id",
    "compound_name", "cultivar_source", "source_publication", "source_attempted",
    "structure_status", "reason", "final_score", "ligand_only_score",
    "ligand_only_rank", "failed_rules", "pattern", "pair_overlap",
    "sensitivity_label", "N_star", "U_star", "notes",
]
diag_out = (
    pd.concat([filtered, disagreement, sensitivity, ledger], ignore_index=True)
    .reindex(columns=cols)
    .fillna("")
)
diag_out.to_csv(results / "diagnostics.csv", index=False)
print(f"[3] diagnostics: {len(diag_out)} rows  (was {len(pd.read_csv(results / 'diagnostics.csv'))}, "
      f"dropped {len(pd.read_csv(results / 'diagnostics.csv')) - len(diag_out)} blanks/dupes)")

[1] compound_level: 5 rows  (was 5)
[2] metadata:
{
  "N_star": 10,
  "U_star": 5,
  "shortlist_pair_n": 10,
  "screened_compound_n": 28,
  "known_active_count": 0,
  "high_ml_ad_extrapolation_count": 5,
  "ad_extrapolation_count": 5,
  "ad_missing_count": 0,
  "docking_missing_count": 0
}
[3b] filtered_but_interesting: 4 rows
[3] diagnostics: 36 rows  (was 36, dropped 0 blanks/dupes)
